In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [6]:
!pip install dagshub mlflow scikit-learn xgboost pandas numpy --quiet

In [7]:
import dagshub, mlflow, mlflow.sklearn
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

dagshub.init(repo_owner='lshek22', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

Initialized MLflow to track repo "lshek22/IEEE-CIS-Fraud-Detection"

Repository lshek22/IEEE-CIS-Fraud-Detection initialized!

In [8]:


best_model = mlflow.sklearn.load_model("models:/IEEEFraud_BestModel/1")
print("got best model")

got best model


In [10]:
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv').merge(
       pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv'), on='TransactionID', how='left')

test['TransactionAmt_log'] = np.log1p(test['TransactionAmt'])
test['Transaction_hour'] = (test['TransactionDT'] / 3600) % 24
test['Transaction_day']  = (test['TransactionDT'] / (3600 * 24)) % 7

train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')

for col in ['card1', 'card2', 'addr1']:
    freq = train[col].value_counts() / len(train)     
    test[f'{col}_freq'] = test[col].map(freq).fillna(0)     

test_ids = test['TransactionID']
preds = best_model.predict_proba(test)[:, 1]

submission = pd.DataFrame({'TransactionID': test_ids, 'isFraud': preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   TransactionID   isFraud
0        3663549  0.026337
1        3663550  0.095144
2        3663551  0.088505
3        3663552  0.067790
4        3663553  0.167055
